# VAR -- Vector Autoregression (Pipeline B)

**Model**: VAR via `statsmodels.tsa.vector_ar.VAR`  
**Target**: `gdpc1`  
**Variables**: Cat2 (outbs, outnfb, gcec1, houstne + COVID = 7 total)  
**Train**: 1959-2007 / **Val**: 2008-2016 / **Test**: 2017-2025  
**Scaling**: NO (equivariant under linear transformation)  
**HP tuning**: NO (lag order selected by AIC inside fit)  
**n_lags**: 3 (for flatten_data -- creates lagged copies)  
**VAR lags**: 1 (endogenous VAR lag order, separate from n_lags)  
**Target shift**: gdpc1 shifted by 3 months to align quarterly values  
**cols_to_drop**: Constant columns (nunique==1) dropped dynamically

In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.api import VAR as SM_VAR
from scipy.stats import norm

plt.rcParams["figure.figsize"] = [15, 10]

sys.path.insert(0, os.path.join(os.path.pardir, "data"))
from helpers import (gen_lagged_data, flatten_data, mean_fill_dataset,
                     get_features, load_data, PREDICTIONS_DIR, TARGET, COVID)

SEED = 42
np.random.seed(SEED)

TRAIN_START = "1959-01-01"
TRAIN_END   = "2007-12-31"
VAL_END     = "2016-12-31"
TEST_START  = "2017-01-01"
TEST_END    = "2025-12-31"
N_LAGS      = 3   # for flatten_data
VAR_LAGS    = 1   # endogenous VAR lag order
VINTAGES    = {"m1": -2, "m2": -1, "m3": 0}

print("VAR -- Cat2, n_lags={}, VAR lags={}, backend=statsmodels".format(N_LAGS, VAR_LAGS))

VAR -- Cat2, n_lags=3, VAR lags=1, backend=statsmodels


In [2]:
monthly, _, metadata = load_data()
features = get_features("cat2", with_covid=True)
print("Features: {}".format(features))

# Compute cols_to_drop once from full pre-flattened training data.
# Columns with nunique==1 are constant (e.g. COVID dummies in pre-2020)
# and break VAR estimation.
pre_train = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= TRAIN_END)].reset_index(drop=True)
# Drop columns entirely NaN in training window (e.g. ppifis starts 2010)
all_nan_cols = [c for c in pre_train.columns if c != "date" and pre_train[c].notna().sum() == 0]
pre_train = pre_train.drop(columns=all_nan_cols)
pre_train_filled = mean_fill_dataset(pre_train, pre_train)
pre_train_flat = flatten_data(pre_train_filled, TARGET, N_LAGS)
pre_train_flat = pre_train_flat.loc[pre_train_flat.date.dt.month.isin([3, 6, 9, 12]), :]
pre_train_flat = pre_train_flat.dropna(axis=0, how="any").reset_index(drop=True)

# Filter to Cat2 columns
all_cols = [c for c in pre_train_flat.columns if c != "date"]
cat2_cols = [c for c in all_cols if any(c == f or c.startswith(f + "_") for f in features)
             and c != TARGET]
nunique = pre_train_flat[cat2_cols].nunique()
cols_to_drop = nunique[nunique <= 1].index.tolist()
cat2_cols = [c for c in cat2_cols if c not in cols_to_drop]

print("Cat2 flattened cols: {} used, {} dropped (constant)".format(
    len(cat2_cols), len(cols_to_drop)))
if cols_to_drop:
    print("  Dropped: {}".format(cols_to_drop))

Features: ['outbs', 'outnfb', 'gcec1', 'houstne', 'covid_2020q2', 'covid_2020q3', 'covid_2020q4']


Cat2 flattened cols: 10 used, 18 dropped (constant)
  Dropped: ['covid_2020q2', 'covid_2020q3', 'covid_2020q4', 'gcec1_1', 'outnfb_1', 'outbs_1', 'covid_2020q2_1', 'covid_2020q3_1', 'covid_2020q4_1', 'gcec1_2', 'outnfb_2', 'outbs_2', 'covid_2020q2_2', 'covid_2020q3_2', 'covid_2020q4_2', 'covid_2020q2_3', 'covid_2020q3_3', 'covid_2020q4_3']


In [3]:
# Rolling test loop
test_quarters = monthly[(monthly["date"] >= TEST_START) &
                         (monthly["date"] <= TEST_END) &
                         monthly["date"].dt.month.isin([3, 6, 9, 12])]["date"].tolist()

predictions = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual = monthly[monthly["date"] == pd_q][TARGET].values[0]
    actuals_list.append(actual)

    for vn, offset in VINTAGES.items():
        fc = pd_q + pd.DateOffset(months=offset)

        # Step 1: ragged-edge mask
        train = monthly[(monthly["date"] >= TRAIN_START) & (monthly["date"] <= fc)].reset_index(drop=True)
        train_m = gen_lagged_data(metadata, train.copy(), fc, lag=0)

        # Step 2: shift target by 3 (align quarterly GDP with next Q-end)
        train_m[TARGET] = train_m[TARGET].shift(3)
        # Fill any NaN introduced by shift at the target date
        train_m.loc[train_m["date"] == fc, TARGET] = np.nanmean(train_m[TARGET])

        # Step 3: mean-fill
        train_f = mean_fill_dataset(train_m, train_m)

        # Step 4: flatten
        train_fl = flatten_data(train_f, TARGET, N_LAGS)
        train_fl = train_fl.loc[train_fl.date.dt.month.isin([3, 6, 9, 12]), :]
        train_fl = train_fl.dropna(axis=0, how="any").reset_index(drop=True)

        # Filter to Cat2 feature columns (excluding dropped constants)
        feature_cols = [c for c in cat2_cols if c in train_fl.columns]

        if len(train_fl) < 10 or len(feature_cols) < 2:
            predictions[vn].append(np.nan)
            continue

        try:
            # Build VAR dataset: must include target + features
            var_cols = [TARGET] + feature_cols
            var_data = train_fl[var_cols].copy()
            var_data = mean_fill_dataset(var_data, var_data)

            model = SM_VAR(var_data.values)
            results = model.fit(VAR_LAGS, trend="c")
            # Forecast 1 step ahead, get target column (first column)
            fcst = results.forecast(var_data.values[-VAR_LAGS:], steps=1)
            target_col = list(var_data.columns).index(TARGET)
            pred = fcst[0, target_col]
        except Exception:
            pred = np.nan
            failed += 1

        predictions[vn].append(pred)

    if (i + 1) % 8 == 0:
        print("  {} / {}".format(i + 1, len(test_quarters)))

print("Done. {} quarters, {} failures.".format(len(test_quarters), failed))

  8 / 36


  16 / 36


  24 / 36


  32 / 36


Done. 36 quarters, 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)
for vn in VINTAGES:
    pd.DataFrame({
        "date": test_quarters, "actual": actuals_list,
        "prediction": predictions[vn],
    }).to_csv(os.path.join(PREDICTIONS_DIR, "var_{}.csv".format(vn)), index=False)
    print("Saved var_{}.csv".format(vn))

Saved var_m1.csv
Saved var_m2.csv
Saved var_m3.csv


In [5]:
def rmse(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[m] - p[m]) ** 2)) if m.sum() > 0 else np.nan

def mae(a, p):
    m = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[m] - p[m])) if m.sum() > 0 else np.nan

panels = {
    "Pre-COVID  (2017-2019)": ("2017-01-01", "2019-12-31"),
    "COVID      (2020-2021)": ("2020-04-01", "2021-12-31"),
    "Post-COVID (2022-2025)": ("2022-01-01", "2025-12-31"),
    "Full test  (2017-2025)": ("2017-01-01", "2025-12-31"),
}
a = np.array(actuals_list)
d = pd.to_datetime(test_quarters)
print("VAR backend: statsmodels")
for pn, (ps, pe) in panels.items():
    m = (d >= ps) & (d <= pe)
    print("--- {} ---".format(pn))
    for vn in VINTAGES:
        pp = np.array(predictions[vn])
        print("  {}  RMSFE={:.6f}  MAE={:.6f}  N={}".format(
            vn, rmse(a[m], pp[m]), mae(a[m], pp[m]), m.sum()))

VAR backend: statsmodels
--- Pre-COVID  (2017-2019) ---
  m1  RMSFE=0.002750  MAE=0.002161  N=12
  m2  RMSFE=0.002748  MAE=0.002165  N=12
  m3  RMSFE=0.003320  MAE=0.002879  N=12
--- COVID      (2020-2021) ---
  m1  RMSFE=0.043098  MAE=0.027762  N=7
  m2  RMSFE=0.043102  MAE=0.027736  N=7
  m3  RMSFE=0.063305  MAE=0.045414  N=7
--- Post-COVID (2022-2025) ---
  m1  RMSFE=0.004357  MAE=0.003310  N=16
  m2  RMSFE=0.004348  MAE=0.003280  N=16
  m3  RMSFE=0.004836  MAE=0.003775  N=16
--- Full test  (2017-2025) ---
  m1  RMSFE=0.019593  MAE=0.008161  N=36
  m2  RMSFE=0.019594  MAE=0.008144  N=36
  m3  RMSFE=0.028388  MAE=0.012060  N=36
